# Phase 3 — GDELT Extraction Pipeline

**Notebook:** `colab_03_gdelt_extraction.ipynb`
**Source:** `src/data/gdelt.py`
**Config:** `config/gdelt_queries.yaml`, `config/source_groups.yaml`
**Audit:** `docs/colab_03_setup.md`

**Wall time:** 4-6 hours (mostly Cell 4: extraction + Cell 5: dedup).

**What this does:**
1. Extract multilingual articles from GDELT DOC 2.0 (2022-09-29 → 2026-06-21)
2. Deduplicate with MinHash + LSH
3. Classify by source group (Ukrainian / Russian / Western / Other)
4. Aggregate to daily counts per source group
5. Sample 100 articles for manual precision audit

**How to run:**
1. Runtime → Change runtime type → Python 3, High-RAM
2. Runtime → Run all
3. Wait 4-6 hours
4. Download outputs from `/content/drive/MyDrive/war_signals_phase3/`

## Cell 1: Setup

Mounts Google Drive, installs dependencies, sets paths.

In [ ]:
# Cell 1: Setup
import os
import sys
import subprocess
from pathlib import Path

# Mount Drive (skipped in local mode; set GDRIVE_PROJECT manually)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_ROOT = Path('/content/drive/MyDrive')
except ImportError:
    # Local mode (running outside Colab)
    GDRIVE_ROOT = Path('/home/mykyta/Desktop/katya/WarSignalsThesis/data/news_colab_sim')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

GDRIVE_PROJECT = GDRIVE_ROOT / 'war_signals_phase3'
GDRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

# Clone or update the repo
PROJECT_NAME = 'WarSignalsThesis'
LOCAL_REPO = Path(f'/content/{PROJECT_NAME}')
if not LOCAL_REPO.exists() and not (Path.cwd().name == PROJECT_NAME):
    print(f'Cloning repository to {LOCAL_REPO}...')
    REPO_URL = 'https://github.com/YOUR-USERNAME/WarSignalsThesis.git'
    subprocess.run(['git', 'clone', REPO_URL, str(LOCAL_REPO)], check=True)
elif not LOCAL_REPO.exists() and Path.cwd().name == PROJECT_NAME:
    LOCAL_REPO = Path.cwd()

if LOCAL_REPO.exists():
    os.chdir(LOCAL_REPO)
sys.path.insert(0, str(LOCAL_REPO))

# Install dependencies
print('\nInstalling dependencies...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pandas>=2.0', 'numpy>=1.24', 'pyyaml>=6.0', 'requests',
    'langdetect', 'datasketch', 'tqdm', 'matplotlib'
], check=True)

print(f'\n✓ Setup complete. CWD: {os.getcwd()}')
print(f'  - GDRIVE_PROJECT: {GDRIVE_PROJECT}')

## Cell 2: Load Configuration

Reads the YAML files and prints a summary.

In [ ]:
# Cell 2: Load configuration
import yaml
from pathlib import Path

with open('config/gdelt_queries.yaml') as f:
    queries_cfg = yaml.safe_load(f)
with open('config/source_groups.yaml') as f:
    groups_cfg = yaml.safe_load(f)

print("=== GDELT queries ===")
for q in queries_cfg['queries']:
    print(f"  - {q['name']}: {q['description'].strip()[:80]}")

print(f"\n=== Source groups ===")
for g, info in groups_cfg['groups'].items():
    n = len(info.get('domains', []))
    print(f"  - {g}: {n} domains — {info['description'][:60]}")

print(f"\n=== Output config ===")
for k, v in queries_cfg.get('output', {}).items():
    print(f"  - {k}: {v}")

## Cell 3: Smoke Test

Validates the pipeline on 1 day of data. **If this fails, STOP and debug before running Cell 4.**

In [ ]:
# Cell 3: Smoke test
import json
import time
import requests
from src.data.gdelt import (
    fetch_gdelt_window, classify_all_articles, build_news_daily,
    dedupe_articles, build_gdelt_query_url, _flatten_keywords
)
import pandas as pd

print("=== SMOKE TEST ===\n")
print(f"Date tested: 2024-01-15 (1 day, 4 queries)")
print(f"API sleep: 6 seconds between calls\n")

# Try each query, use the first one that returns articles
all_articles = []
used_query_name = None
for q_idx, q in enumerate(queries_cfg['queries']):
    print(f"--- Query {q_idx+1}/{len(queries_cfg['queries'])}: {q['name']} ---")
    keywords_any = _flatten_keywords(q.get('keywords_any', {}))
    keywords_weapon_any = _flatten_keywords(q.get('keywords_weapon_any', {}))
    print(f"  keywords: {len(keywords_any)} country/actor + {len(keywords_weapon_any)} weapon")
    print(f"  languages: {q.get('languages', [])}")
    
    # Build the URL with proper date format (GDELT requires YYYYMMDDHHMMSS)
    url = build_gdelt_query_url(
        keywords_any=keywords_any,
        keywords_weapon_any=keywords_weapon_any,
        languages=q.get('languages', []),
        start="2024-01-15 00:00:00",
        end="2024-01-15 23:59:59",
    )
    
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 429:
            print(f"  HTTP 429 (rate-limited). Body: {r.text[:200]}")
            print(f"  -> Stopping smoke test. Cell 4 will retry with backoff.")
            raise SystemExit(0)
        elif r.status_code != 200:
            print(f"  HTTP {r.status_code}: {r.text[:200]}")
            continue
        try:
            data = r.json()
        except Exception as e:
            print(f"  JSON decode error: {e}")
            print(f"  Body (first 200): {r.text[:200]}")
            continue
        articles = data.get('articles', []) if isinstance(data, dict) else []
        print(f"  Retrieved {len(articles)} articles")
        if articles:
            all_articles = articles
            used_query_name = q['name']
            print(f"  OK Using this query for full extraction.")
            break
        # Empty result - show URL for debugging
        print(f"  URL (first 300): {url[:300]}")
    except Exception as e:
        print(f"  Error: {e}")
    print()
    time.sleep(6)  # respect rate limit between queries

if not all_articles:
    print("=" * 60)
    print("SMOKE TEST FAILED: 0 articles from all queries.")
    print("=" * 60)
    print("Diagnosis steps:")
    print("  1. Check if any query returned HTTP 429 (rate-limited).")
    print("  2. If rate-limited, wait 5-10 minutes and re-run.")
    print("  3. Otherwise, the queries may be too narrow for 2024-01-15.")
    print()
    print("Cell 4 has its own exponential backoff (4-64s) on 429 errors,")
    print("so you can skip this smoke test and go directly to Cell 4.")
    raise SystemExit(0)

df = pd.DataFrame(all_articles)
print(f"\n--- Validation ---")
print(f"Total articles: {len(df)}")
print(f"Columns: {list(df.columns)[:10]}")
print(f"\nSample article:")
sample = all_articles[0]
for k in ['title', 'domain', 'language', 'sourceCommonName', 'url', 'seendate']:
    if k in sample:
        v = str(sample[k])[:80]
        print(f"  {k}: {v}")

# Classify and aggregate
df = classify_all_articles(df)
print(f"\nSource groups: {df['source_group'].value_counts().to_dict()}")

# Test dedup (won't dedup much with one day, but verifies the function works)
df_dedup = dedupe_articles(df, threshold=0.7)
print(f"\nDedup: {len(df)} -> {len(df_dedup)} articles ({len(df_dedup)/len(df)*100:.1f}% kept)")

print(f"\nOK Smoke test passed using query '{used_query_name}'.")

## Cell 4: Full Extraction (2-4 hours)

Fetches all articles for all queries in monthly windows. Resumable — saves after each (query, month).

In [ ]:
# Cell 4: Full extraction
import time
from src.data.gdelt import fetch_gdelt_window
from pathlib import Path
import pandas as pd
import yaml

RAW_DIR = GDRIVE_PROJECT / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

START = '2022-09-29'
END = '2026-06-21'
API_SLEEP = 5.0  # seconds between calls (GDELT public API rate limit)

months = pd.date_range(START, END, freq='MS').strftime('%Y-%m').tolist()
months.append(END[:7])
months = sorted(set(months))  # unique, sorted

print(f"=== FULL EXTRACTION ===")
print(f"Date range: {START} → {END}")
print(f"Number of monthly windows: {len(months)}")
print(f"Number of queries: {len(queries_cfg['queries'])}")
print(f"Total API calls: {len(months) * len(queries_cfg['queries'])}")
print(f"API sleep: {API_SLEEP}s per call")
print(f"Estimated wall time: ~{(len(months) * len(queries_cfg['queries']) * API_SLEEP) / 60:.0f} minutes (best case, no errors)")
print()

all_records = []
t0 = time.time()
errors = []

for q_idx, q in enumerate(queries_cfg['queries']):
    print(f"\n[{q_idx+1}/{len(queries_cfg['queries'])}] Query: {q['name']}")
    for m_idx, m in enumerate(months):
        out_file = RAW_DIR / f"raw_{q['name']}_{m}.parquet"
        if out_file.exists():
            df_existing = pd.read_parquet(out_file)
            if not df_existing.empty:
                print(f"  [SKIP] {m}: {len(df_existing)} cached")
                all_records.extend(df_existing.to_dict('records'))
                continue
        month_start = pd.Timestamp(m).date()
        month_end = (pd.Timestamp(m) + pd.offsets.MonthEnd(0)).date()
        if month_start < pd.Timestamp(START).date():
            month_start = pd.Timestamp(START).date()
        if month_end > pd.Timestamp(END).date():
            month_end = pd.Timestamp(END).date()
        elapsed = time.time() - t0
        print(f"  [FETCH] {month_start} → {month_end} ({elapsed/60:.1f}m elapsed)", end=" ", flush=True)
        try:
            articles = fetch_gdelt_window(
                q,
                start=month_start.isoformat(),
                end=month_end.isoformat(),
                api_sleep=API_SLEEP,
            )
            if articles:
                pd.DataFrame(articles).to_parquet(out_file)
            else:
                pd.DataFrame().to_parquet(out_file)  # empty marker
            all_records.extend(articles)
            print(f"→ {len(articles)} articles")
        except Exception as e:
            print(f"ERROR: {e}")
            errors.append((q['name'], m, str(e)))
            continue

print(f"\n=== EXTRACTION COMPLETE ===")
print(f"Total articles retrieved: {len(all_records)}")
print(f"Wall time: {(time.time() - t0)/60:.1f} minutes")
if errors:
    print(f"Errors: {len(errors)}")
    for q_name, m, err in errors[:5]:
        print(f"  - {q_name} {m}: {err}")

## Cell 5: Deduplication (1-2 hours)

MinHash + LSH on article titles. Removes near-duplicates (syndicated copies).

In [ ]:
# Cell 5: Deduplication
import pandas as pd
import time
from pathlib import Path
from src.data.gdelt import dedupe_articles

RAW_DIR = GDRIVE_PROJECT / 'raw'
DEDUP_FILE = GDRIVE_PROJECT / 'gdelt_articles_dedup.parquet'

print("=== DEDUPLICATION ===")
t0 = time.time()

# Load all raw files
files = sorted(RAW_DIR.glob('raw_*.parquet'))
print(f"Found {len(files)} raw files")
dfs = []
for f in files:
    df = pd.read_parquet(f)
    if not df.empty:
        dfs.append(df)
if not dfs:
    raise RuntimeError("No raw files found. Did Cell 4 run successfully?")
raw = pd.concat(dfs, ignore_index=True)
print(f"Total articles loaded: {len(raw)}")
print(f"Unique URLs: {raw['url'].nunique() if 'url' in raw.columns else 'N/A'}")

# Title field may be named differently
title_col = None
for cand in ['title', 'title_translated', 'socialtitle']:
    if cand in raw.columns:
        title_col = cand
        break
if title_col is None:
    raise RuntimeError(f"No title field found. Columns: {list(raw.columns)[:10]}")
print(f"Using title field: {title_col}")

# Dedupe
deduped = dedupe_articles(raw, title_col=title_col, threshold=0.7, num_perm=128)
print(f"\nDedup: {len(raw)} → {len(deduped)} ({len(deduped)/len(raw)*100:.1f}% kept)")
print(f"Wall time: {(time.time() - t0)/60:.1f} minutes")

# Save
deduped.to_parquet(DEDUP_FILE)
print(f"\n✓ Saved {DEDUP_FILE} ({DEDUP_FILE.stat().st_size / 1024 / 1024:.1f} MB)")

## Cell 6: Classification (5-10 minutes)

Adds `source_group` and detected `language` columns.

In [ ]:
# Cell 6: Classification
import pandas as pd
from src.data.gdelt import classify_all_articles, detect_language
from tqdm import tqdm

DEDUP_FILE = GDRIVE_PROJECT / 'gdelt_articles_dedup.parquet'
CLASS_FILE = GDRIVE_PROJECT / 'gdelt_articles_classified.parquet'

print("=== CLASSIFICATION ===")
df = pd.read_parquet(DEDUP_FILE)
print(f"Loaded {len(df)} articles")

# Source group classification (uses config/source_groups.yaml)
df = classify_all_articles(df)
print(f"\nSource groups:\n{df['source_group'].value_counts()}")

# Language detection (only if not already present, or rerun to validate)
if 'language' not in df.columns or df['language'].isna().mean() > 0.5:
    print("\nDetecting languages (this takes a few minutes)...")
    titles = df['title'].fillna('').astype(str)
    df['language'] = [detect_language(t) for t in tqdm(titles)]
else:
    print("\nLanguage field already populated; using existing values.")
print(f"\nLanguages:\n{df['language'].value_counts().head(10)}")

# Save
df.to_parquet(CLASS_FILE)
print(f"\n✓ Saved {CLASS_FILE} ({CLASS_FILE.stat().st_size / 1024 / 1024:.1f} MB)")

In [ ]:
# Cell 7: Daily aggregation
import pandas as pd
from src.data.gdelt import build_news_daily

CLASS_FILE = GDRIVE_PROJECT / 'gdelt_articles_classified.parquet'
DAILY_FILE = GDRIVE_PROJECT / 'news_daily.parquet'

print("=== DAILY AGGREGATION ===")
df = pd.read_parquet(CLASS_FILE)
print(f"Loaded {len(df)} articles")

# GDELT's date field is 'seendate' in YYYYMMDDTHHMMSSZ format
# Normalize it to a date column for grouping
if 'seendate' in df.columns:
    df['date'] = pd.to_datetime(df['seendate'], format='%Y%m%dT%H%M%SZ', errors='coerce').dt.normalize()
    # Drop rows where date parsing failed
    df = df.dropna(subset=['date'])
    print(f"Articles with valid dates: {len(df)}")
elif 'date' not in df.columns:
    raise RuntimeError("Neither 'seendate' nor 'date' column found in classified data")

daily = build_news_daily(df, date_col='date', group_col='source_group')
print(f"\nDaily aggregate: {daily.shape}")
print(f"Date range: {daily.index.min().date()} to {daily.index.max().date()}")
print(f"\nSummary:")
print(daily.describe().round(1))

daily.to_parquet(DAILY_FILE)
daily.to_csv(DAILY_FILE.with_suffix('.csv'))
print(f"\n✓ Saved {DAILY_FILE}")


## Cell 8: Summary

Prints coverage statistics, top sources, and final file list.

In [ ]:
# Cell 8: Summary
import pandas as pd
from pathlib import Path

CLASS_FILE = GDRIVE_PROJECT / 'gdelt_articles_classified.parquet'
DAILY_FILE = GDRIVE_PROJECT / 'news_daily.parquet'
AUDIT_FILE = GDRIVE_PROJECT / 'manual_precision_audit.csv'

print("=" * 70)
print("PHASE 3 — GDELT EXTRACTION — SUMMARY")
print("=" * 70)

df = pd.read_parquet(CLASS_FILE)
print(f"\nTotal articles (after dedup): {len(df):,}")

# Normalize date field
if 'seendate' in df.columns:
    df['date'] = pd.to_datetime(df['seendate'], format='%Y%m%dT%H%M%SZ', errors='coerce').dt.normalize()
    df = df.dropna(subset=['date'])
    print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
elif 'date' in df.columns:
    print(f"Date range: {pd.to_datetime(df['date']).min().date()} → {pd.to_datetime(df['date']).max().date()}")

print(f"\nSource groups:\n{df['source_group'].value_counts().to_string()}")

print(f"\nTop 10 domains (Ukrainian):")
ukr = df[df['source_group'] == 'ukrainian']
if not ukr.empty and 'domain' in ukr.columns:
    print(ukr['domain'].value_counts().head(10).to_string())

print(f"\nTop 10 domains (Russian):")
ru = df[df['source_group'] == 'russian']
if not ru.empty and 'domain' in ru.columns:
    print(ru['domain'].value_counts().head(10).to_string())

print(f"\nTop 10 domains (Western):")
west = df[df['source_group'] == 'western']
if not west.empty and 'domain' in west.columns:
    print(west['domain'].value_counts().head(10).to_string())

# Manual audit
from src.data.gdelt import manual_precision_audit
audit = manual_precision_audit(df, n_per_group=25, seed=42)
audit.to_csv(AUDIT_FILE, index=False)
print(f"\nManual precision audit sample saved: {AUDIT_FILE}")
print(f"  {len(audit)} articles to label (25 per group × 4 groups)")
print(f"  Open the CSV in Google Sheets or Excel; fill in the 'relevant' column (1 or 0)")

daily = pd.read_parquet(DAILY_FILE)
print(f"\nDaily aggregate: {daily.shape[0]} days × {daily.shape[1]} columns")
print(f"  - Total articles: {daily['n_articles_total'].sum():,}")
print(f"  - Days with at least 1 article: {(daily['n_articles_total'] > 0).sum()}")

print(f"\n=== FILES (download from /content/drive/MyDrive/war_signals_phase3/) ===")
for f in sorted(GDRIVE_PROJECT.iterdir()):
    if f.is_file():
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  - {f.name} ({size_mb:.1f} MB)")
print(f"\n{'='*70}")
print("DONE! Copy these files to your local data/interim/news/ and data/processed/news/.")
print("Then run the local post-processing script (Phase 3 audit + figures).")
print(f"{'='*70}")
